# Cairo: proving termination with the DSL

Model the program below as a `dslModule`, load its ranking function `V`, and use Z3 to
verify the two termination conditions: `V >= 0`, and `V` strictly decreases on every loop
iteration.

## The program

```c
int main() {
    int x;
    x = __VERIFIER_nondet_int();
    if (x > 0) {
        while (x != 0) {
            x = x - 1;
        }
    }
    return 0;
}
```

The loop is entered only when `x > 0` and decrements `x` to `0`. `x` is modelled over the
integers (`LIA`); the program terminates over the integers but not the rationals.

In [1]:
import json
from pathlib import Path

import torch
import torch.nn as nn
import z3

from zrth import LIA, Sort, Wire
from zrth.dsl import dslModule, nxt, ite, ne
from zrth.torch import Module
from zrth import z3 as zz3

## The program as a `dslModule`

`x` is a `(latched, next)` wire pair. Its nondeterministic initial value is an external
input `x0`, so `init` returns `nxt(x0)`; the loop body is the transition
`x' = ite(x != 0, x - 1, x)`.

In [2]:
class Cairo(dslModule):
    def init(self, extl):
        x0 = extl
        return nxt(x0)                     # nondeterministic initial value

    def update(self, ctrl):
        x = ctrl
        return ite(ne(x, 0), x - 1, x)     # while x != 0: x = x - 1


INT = Sort.Int([1, 1])
x = (Wire(INT), Wire(INT))     # (latched, next)
x0 = (Wire(INT), Wire(INT))
program = Cairo(theory=LIA, ctrl=(x,), extl=(x0,))
print(program)

module
  external
    w2, w3 : Int(1, 1)
  interface
    w0, w1 : Int(1, 1)
  atom controls w1 reads w0 awaits w3
  init
    Id w1; w3
  update
    [[0]]
Tensor[[1, 1], Int64] w4; 
    Ne w5; w0, w4
    [[1]]
Tensor[[1, 1], Int64] w6; 
    Sub w7; w0, w6
    Ite w8; w5, w7, w0
    Id w1; w8



`x` is controlled and `x0` is external (never driven here), so the module is open.
`update` is the loop transition.

## The ranking function

`Cairo.ranking_function.json` stores `V` as dense layers (`W`, `b`) with a ReLU between
them, plus `delta_threshold` (the minimum decrease per step). We rebuild it as an
`nn.Module` and wrap it with `zrth.torch.Module`.

In [3]:
# resolve the JSON whether cwd is this folder or the repo root
_name = "Cairo.ranking_function.json"
rf_path = next(d / _name for d in (Path.cwd(), Path.cwd() / "tutorials" / "cairo") if (d / _name).exists())
rf = json.loads(rf_path.read_text())
layers = rf["layers"]
delta = rf["delta_threshold"]


class Ranking(nn.Module):
    """Linear -> ReLU -> Linear, with shapes and weights from the JSON."""

    def __init__(self, layers):
        super().__init__()
        (o1, i1), (o2, i2) = [(len(L["W"]), len(L["W"][0])) for L in layers]
        self.fc1 = nn.Linear(i1, o1)
        self.fc2 = nn.Linear(i2, o2)
        with torch.no_grad():
            self.fc1.weight.copy_(torch.tensor(layers[0]["W"]))
            self.fc1.bias.copy_(torch.tensor(layers[0]["b"]))
            self.fc2.weight.copy_(torch.tensor(layers[1]["W"]))
            self.fc2.bias.copy_(torch.tensor(layers[1]["b"]))

    def forward(self, s):
        return self.fc2(torch.relu(self.fc1(s)))


V = Module(Ranking(layers))
print(V)

module
  external
    w9, w10 : Real(1, 1)
  interface
    w11, w12 : Real(1, 1)
  atom controls w12 awaits w10
  init
    Transpose w13; w10
    Linear w14; w13
    Transpose w15; w14
    ReLU w16; w15
    Transpose w17; w16
    Linear w18; w17
    Transpose w19; w18
    Id w12; w19
  update
    Transpose w13; w10
    Linear w14; w13
    Transpose w15; w14
    ReLU w16; w15
    Transpose w17; w16
    Linear w18; w17
    Transpose w19; w18
    Id w12; w19



## Verification

Negate each condition and ask Z3 for a counterexample; `unsat` means it holds for every
state. `x'` comes from the program's `update`; `V` is evaluated on `x` and `x'` (cast to
real). The domain is the in-loop condition `x >= 1`.

In [4]:
# x' from the program's update block
x_lat, x_nxt = list(program.ctrl)[0]
state = {x_lat: [z3.Int("x")]}
for atom in program.atoms:
    for term in atom.update:
        state.update(zip(term.write, zz3.eval(term.itype, [state[w] for w in term.read])))

x_sym = state[x_lat][0]
print("x' =", state[x_nxt][0])


# V as a Z3 expression, by executing the ranking module's terms
extl, intf = list(V.extl), list(V.intf)

def eval_V(inp):
    r = {extl[0][1]: inp}
    for atom in V.atoms:
        for term in atom.update:
            r.update(zip(term.write, zz3.eval(term.itype, [r[w] for w in term.read])))
    return r[intf[0][1]][0]


V_s = eval_V([z3.ToReal(x_sym)])
V_s_next = eval_V([z3.ToReal(state[x_nxt][0])])
domain = [x_sym >= 1]

x' = If(0 != x, x - 1, x)


**Condition 1** — `V(x) >= 0`.

In [5]:
solver = z3.Solver()
solver.add(*domain)
solver.add(V_s < 0)                        # negate
if solver.check() == z3.unsat:
    print("VERIFIED: V(x) >= 0 for all in-loop states")
else:
    print("COUNTEREXAMPLE:", solver.model())

VERIFIED: V(x) >= 0 for all in-loop states


**Condition 2** — `V(x) - V(x') >= delta`.

In [6]:
solver = z3.Solver()
solver.add(*domain)
solver.add(V_s - V_s_next < delta)         # negate
if solver.check() == z3.unsat:
    print(f"VERIFIED: V(x) - V(x') >= {delta} at every in-loop step")
else:
    print("COUNTEREXAMPLE:", solver.model())

VERIFIED: V(x) - V(x') >= 1.0 at every in-loop step


## Result

Both conditions hold for every reachable state, so `V` is a ranking function and the loop
terminates.